# Advanced: Multi-Strategy Field Operations with PAMOLA.CORE

**Goal**: Master all 3 field manipulation strategies using operation.execute()

**What you'll learn:**
- **Strategy 1**: Lookup-based enrichment from external dictionaries
- **Strategy 2**: Conditional logic for dynamic field creation
- **Strategy 3**: Expression-based calculations for derived fields
- Calculate field correlation metrics
- Analyze data enrichment impact
- Production deployment patterns

**Prerequisites:**
- Completed the simple notebook
- Understanding of data transformation concepts
- Familiarity with operation.execute() pattern
- Python 3.8+
- PAMOLA.CORE installed (auto-installs if needed)

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── transformations/field_ops/
        └── 02_add_modify_fields_advanced.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os
from pathlib import Path
from datetime import datetime
import time
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print("   Repository: https://github.com/DGT-Network/PAMOLA.git")
        print("   Branch: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.transformations.field_ops.add_modify_fields import (
        AddOrModifyFieldsOperation
    )
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Complex Dataset

**How to use:**
- Run to load or generate 1000-record dataset
- Auto-creates sample if file not found
- Review data structure before proceeding

**What you'll see:**
- Load status (from file or generated)
- Dataset overview (1000 records, 6 columns)
- Sample records (first 5 rows)
- Column statistics (types, ranges, unique counts)

**Note:** Generated data includes product codes, quantities, prices, regions, customer types suitable for field operations

In [ ]:
# Try to load from file first (in examples directory)
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'add_modify_complex_data.csv'
print(f"📂 Looking for data at: {data_path}\n")

if data_path.exists():
    print(f"📂 Loading data from: {data_path}")
    df = read_csv(data_path)
    print(f"✓ Loaded {len(df)} records from file")
else:
    print("📊 Generating synthetic dataset (1000 records)...\n")
    
    np.random.seed(42)
    n_records = 1000
    
    # Product codes
    products = ['P001', 'P002', 'P003', 'P004', 'P005', 'P006', 'P007', 'P008']
    product_data = np.random.choice(products, n_records)
    
    # Quantities and prices
    quantities = np.random.randint(1, 50, n_records)
    unit_prices = np.random.uniform(10.0, 100.0, n_records).round(2)
    
    # Regions and customer types
    regions = ['North', 'South', 'East', 'West', 'Central']
    region_data = np.random.choice(regions, n_records)
    
    customer_types = ['Premium', 'Standard', 'Basic']
    customer_data = np.random.choice(customer_types, n_records, p=[0.2, 0.5, 0.3])
    
    df = pd.DataFrame({
        'record_id': range(1, n_records + 1),
        'product_code': product_data,
        'quantity': quantities,
        'unit_price': unit_prices,
        'region': region_data,
        'customer_type': customer_data
    })
    
    # Save for future use (with error handling)
    try:
        data_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(data_path, index=False)
        print(f"✓ Generated and saved to: {data_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {data_path} (file may be open)")
        print(f"   Dataset generated in memory only")

print(f"\n📊 Dataset Overview:")
print("=" * 80)
print(f"  Records: {len(df):,}")
print(f"  Columns: {', '.join(df.columns)}")

print(f"\n🔍 Sample Records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:15s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    else:
        print(f"  {col:15s} ({dtype_str:10s}): min={df[col].min()}, max={df[col].max()}")

print(f"\n💡 Perfect dataset for testing all 3 field operation strategies!")

## Step 3: Prepare Lookup Tables

**How to use:**
- Run to check/create lookup table files
- Creates product categories mapping
- Creates region classifications
- Creates customer segment definitions

**What you'll see:**
- File status (✓ found or 🔧 created)
- Lookup table structures with mapping counts
- Sample mappings displayed
- File locations in examples/data_examples/

**Note:** These lookup tables enable data enrichment in Strategy 1

In [ ]:
# Define file paths
examples_dir = project_root / 'examples'
product_lookup_path = examples_dir / 'data_examples' / 'product_categories.json'
region_lookup_path = examples_dir / 'data_examples' / 'region_classifications.json'
customer_lookup_path = examples_dir / 'data_examples' / 'customer_segments.json'

# Check/create product category lookup
if product_lookup_path.exists():
    print(f"✓ Found product categories: {product_lookup_path}")
    with open(product_lookup_path, 'r') as f:
        product_lookup = json.load(f)
    print(f"   Mappings: {len(product_lookup)}")
else:
    print(f"🔧 Creating product categories lookup...")
    product_lookup = {
        'P001': 'Electronics',
        'P002': 'Electronics',
        'P003': 'Furniture',
        'P004': 'Furniture',
        'P005': 'Clothing',
        'P006': 'Clothing',
        'P007': 'Food',
        'P008': 'Food'
    }
    try:
        product_lookup_path.parent.mkdir(parents=True, exist_ok=True)
        with open(product_lookup_path, 'w') as f:
            json.dump(product_lookup, f, indent=2)
        print(f"✓ Created: {product_lookup_path}")
        print(f"   Categories: {set(product_lookup.values())}")
    except PermissionError:
        print(f"⚠️  Cannot save (file may be open)")

# Check/create region classification lookup
if region_lookup_path.exists():
    print(f"\n✓ Found region classifications: {region_lookup_path}")
    with open(region_lookup_path, 'r') as f:
        region_lookup = json.load(f)
    print(f"   Mappings: {len(region_lookup)}")
else:
    print(f"\n🔧 Creating region classifications lookup...")
    region_lookup = {
        'North': 'Urban',
        'South': 'Rural',
        'East': 'Urban',
        'West': 'Suburban',
        'Central': 'Urban'
    }
    try:
        region_lookup_path.parent.mkdir(parents=True, exist_ok=True)
        with open(region_lookup_path, 'w') as f:
            json.dump(region_lookup, f, indent=2)
        print(f"✓ Created: {region_lookup_path}")
        print(f"   Classifications: {set(region_lookup.values())}")
    except PermissionError:
        print(f"⚠️  Cannot save (file may be open)")

# Check/create customer segment lookup
if customer_lookup_path.exists():
    print(f"\n✓ Found customer segments: {customer_lookup_path}")
    with open(customer_lookup_path, 'r') as f:
        customer_lookup = json.load(f)
    print(f"   Mappings: {len(customer_lookup)}")
else:
    print(f"\n🔧 Creating customer segments lookup...")
    customer_lookup = {
        'Premium': 'VIP',
        'Standard': 'Regular',
        'Basic': 'Regular'
    }
    try:
        customer_lookup_path.parent.mkdir(parents=True, exist_ok=True)
        with open(customer_lookup_path, 'w') as f:
            json.dump(customer_lookup, f, indent=2)
        print(f"✓ Created: {customer_lookup_path}")
        print(f"   Segments: {set(customer_lookup.values())}")
    except PermissionError:
        print(f"⚠️  Cannot save (file may be open)")

print("\n💡 All lookup tables ready for Strategy 1!")
print("=" * 80)

## Step 4: Setup Shared Environment

**How to use:**
1. **CUSTOMIZE FIELD_CONFIG**: Define fields for each strategy
   - `strategy1_fields`: Fields for lookup enrichment
   - `strategy2_fields`: Fields for conditional logic
   - `strategy3_fields`: Fields for calculations
2. Run to validate and setup environment

**What you'll see:**
- Strategy field lists validated
- Base column checks (✓ marks)
- Task directory created (✓)
- TaskReporter initialized (✓)
- DataSource and config ready (✓)

**Note:** All base columns must exist before executing strategies

In [ ]:
# Field configuration - CUSTOMIZE THESE!
FIELD_CONFIG = {
    "strategy1_fields": {
        "product_category": "product_code",
        "region_type": "region",
        "customer_segment": "customer_type"
    },
    "strategy2_fields": {
        "discount_tier": ["quantity", "customer_type"],
        "priority_flag": ["customer_type", "region"]
    },
    "strategy3_fields": {
        "total_amount": ["quantity", "unit_price"],
        "volume_category": ["quantity"]
    }
}

# Validate all base columns exist in dataset
print("📋 Validating field configuration...\n")

# Strategy 1: Lookup-based
print("Strategy 1 (Lookup):")
for new_field, base_col in FIELD_CONFIG["strategy1_fields"].items():
    if base_col not in df.columns:
        raise ValueError(f"❌ Base column '{base_col}' not found!")
    print(f"  ✓ {new_field:20s} ← {base_col}")

# Strategy 2: Conditional
print("\nStrategy 2 (Conditional):")
for new_field, base_cols in FIELD_CONFIG["strategy2_fields"].items():
    for base_col in base_cols:
        if base_col not in df.columns:
            raise ValueError(f"❌ Base column '{base_col}' not found!")
    print(f"  ✓ {new_field:20s} ← {', '.join(base_cols)}")

# Strategy 3: Expression-based
print("\nStrategy 3 (Expression):")
for new_field, base_cols in FIELD_CONFIG["strategy3_fields"].items():
    for base_col in base_cols:
        if base_col not in df.columns:
            raise ValueError(f"❌ Base column '{base_col}' not found!")
    print(f"  ✓ {new_field:20s} ← {', '.join(base_cols)}")

# Configuration helper (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset"
    }

# Create task directory (in examples/data_examples)
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'advanced_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="advanced_001",
    task_type="multi_strategy_fields",
    description="Multi-strategy field add/modify operations",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config
kwargs = create_config_kwargs()
print(f"✓ Config kwargs ready")

# Create DataSource
data_source = DataSource(
    dataframes={"main_dataset": df}
)
print("✓ DataSource created")

print(f"\n✅ Shared environment ready for all strategies!")

## STRATEGY 1: Lookup-Based Enrichment

**How to use:**
- Uses external lookup tables for enrichment
- Run to execute lookup-based strategy
- Monitor progress bar (6 steps)

**What you'll see:**
- Configuration summary
- Progress: validation → load lookups → map → metrics → viz → save
- Completion time and status
- Field additions (3 new fields expected)

**Key parameters:**
- `operation_type='add_from_lookup'`: Maps values from lookup tables
- `product_category`: Maps product codes to categories
- `region_type`: Maps regions to classifications
- `customer_segment`: Maps customer types to segments
- `mode='REPLACE'`: Adds fields to original dataframe

**Note:** Best for enrichment from external reference data

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 1: LOOKUP-BASED ENRICHMENT")
print("=" * 80 + "\n")

# Setup progress tracker
tracker_s1 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 1: Lookup-based",
    unit="steps",
    track_memory=True,
    level=0
)

# Load lookup tables
with open(product_lookup_path, 'r') as f:
    product_categories = json.load(f)
with open(region_lookup_path, 'r') as f:
    region_types = json.load(f)
with open(customer_lookup_path, 'r') as f:
    customer_segments = json.load(f)

# Create operation
operation_s1 = AddOrModifyFieldsOperation(
    # Field operations
    field_operations={
        'product_category': {
            'operation_type': 'add_from_lookup',
            'base_on_column': 'product_code',
            'lookup_table_name': 'product_categories'
        },
        'region_type': {
            'operation_type': 'add_from_lookup',
            'base_on_column': 'region',
            'lookup_table_name': 'region_types'
        },
        'customer_segment': {
            'operation_type': 'add_from_lookup',
            'base_on_column': 'customer_type',
            'lookup_table_name': 'customer_segments'
        }
    },
    
    # Lookup tables
    lookup_tables={
        'product_categories': product_categories,
        'region_types': region_types,
        'customer_segments': customer_segments
    },
    
    # Mode configuration
    mode='REPLACE',
    output_format='csv',
    
    # Processing settings
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 1 configured")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

# Execute
result_s1 = operation_s1.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy1_lookup',
    reporter=task_reporter,
    progress_tracker=tracker_s1,
    **kwargs
)

elapsed_s1 = time.time() - start_time
print(f"\n✅ Strategy 1 completed in {elapsed_s1:.2f}s")

# Load results (NEWEST file)
output_files_s1 = sorted(
    list((task_dir / 'strategy1_lookup' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s1:
    result_df_s1 = pd.read_csv(output_files_s1[0])
    
    # Show added fields
    new_fields = [col for col in result_df_s1.columns if col not in df.columns]
    print(f"\n📊 Added {len(new_fields)} fields: {', '.join(new_fields)}")
    for field in new_fields:
        print(f"   {field}: {result_df_s1[field].nunique()} unique values")

## STRATEGY 2: Conditional Logic

**How to use:**
- Creates fields based on complex conditions
- Run to execute conditional strategy
- Monitor progress bar (6 steps)

**What you'll see:**
- Configuration summary
- Progress: validation → evaluate conditions → create fields → metrics → viz → save
- Completion time and status
- Field additions (2 new fields expected)

**Key parameters:**
- `operation_type='add_conditional'`: Applies if-then-else logic
- `discount_tier`: Based on quantity and customer type
- `priority_flag`: Based on customer and region

**Note:** Best for business rules and dynamic categorization

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 2: CONDITIONAL LOGIC")
print("=" * 80 + "\n")

tracker_s2 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 2: Conditional",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s2 = AddOrModifyFieldsOperation(
    # Field operations with conditional logic
    field_operations={
        'discount_tier': {
            'operation_type': 'add_conditional',
            'condition': "row['quantity'] >= 30 and row['customer_type'] == 'Premium'",
            'value_if_true': "'Gold'",
            'value_if_false': "'Silver' if row['quantity'] >= 20 else 'Bronze'"
        },
        'priority_flag': {
            'operation_type': 'add_conditional',
            'condition': "row['customer_type'] == 'Premium' and row['region'] in ['North', 'Central']",
            'value_if_true': "'High'",
            'value_if_false': "'Normal'"
        }
    },
    
    lookup_tables={},
    mode='REPLACE',
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 2 configured")
start_time = time.time()

result_s2 = operation_s2.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy2_conditional',
    reporter=task_reporter,
    progress_tracker=tracker_s2,
    **kwargs
)

elapsed_s2 = time.time() - start_time
print(f"\n✅ Strategy 2 completed in {elapsed_s2:.2f}s")

# Load results
output_files_s2 = sorted(
    list((task_dir / 'strategy2_conditional' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s2:
    result_df_s2 = pd.read_csv(output_files_s2[0])
    
    # Show distribution of new fields
    if 'discount_tier' in result_df_s2.columns:
        print(f"\n📊 Discount Tier Distribution:")
        print(result_df_s2['discount_tier'].value_counts())
    
    if 'priority_flag' in result_df_s2.columns:
        print(f"\n📊 Priority Flag Distribution:")
        print(result_df_s2['priority_flag'].value_counts())

## STRATEGY 3: Expression-Based Calculations

**How to use:**
- Creates calculated fields from expressions
- Run to execute expression-based strategy
- Monitor progress bar (6 steps)

**What you'll see:**
- Configuration summary
- Progress: validation → evaluate expressions → calculate → metrics → viz → save
- Completion time and status
- Field additions (2 new fields expected)

**Key parameters:**
- `operation_type='add_conditional'`: For calculated fields
- `total_amount`: quantity × unit_price
- `volume_category`: Categorize by quantity ranges

**Note:** Optimal for derived metrics and KPI calculations

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 3: EXPRESSION-BASED CALCULATIONS")
print("=" * 80 + "\n")

tracker_s3 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 3: Expression",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s3 = AddOrModifyFieldsOperation(
    # Field operations with expressions
    field_operations={
        'total_amount': {
            'operation_type': 'add_conditional',
            'condition': "True",
            'value_if_true': "row['quantity'] * row['unit_price']",
            'value_if_false': "0"
        },
        'volume_category': {
            'operation_type': 'add_conditional',
            'condition': "row['quantity'] >= 40",
            'value_if_true': "'High'",
            'value_if_false': "'Medium' if row['quantity'] >= 20 else 'Low'"
        }
    },
    
    lookup_tables={},
    mode='REPLACE',
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 3 configured")
start_time = time.time()

result_s3 = operation_s3.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy3_expression',
    reporter=task_reporter,
    progress_tracker=tracker_s3,
    **kwargs
)

elapsed_s3 = time.time() - start_time
print(f"\n✅ Strategy 3 completed in {elapsed_s3:.2f}s")

# Load results
output_files_s3 = sorted(
    list((task_dir / 'strategy3_expression' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s3:
    result_df_s3 = pd.read_csv(output_files_s3[0])
    
    if 'total_amount' in result_df_s3.columns:
        print(f"\n📊 Total Amount Statistics:")
        print(f"   Mean: ${result_df_s3['total_amount'].mean():.2f}")
        print(f"   Median: ${result_df_s3['total_amount'].median():.2f}")
        print(f"   Range: ${result_df_s3['total_amount'].min():.2f} - ${result_df_s3['total_amount'].max():.2f}")
    
    if 'volume_category' in result_df_s3.columns:
        print(f"\n📊 Volume Category Distribution:")
        print(result_df_s3['volume_category'].value_counts())

## Step 5: Strategy Comparison

**How to use:**
- Run AFTER all strategies complete
- Review performance and effectiveness

**What you'll see:**
- Execution times (fastest to slowest)
- Total processing time
- Fields added per strategy
- Data enrichment comparison

**Note:** Compare field addition methods - lookup vs conditional vs expression

In [ ]:
print("\n" + "=" * 80)
print("📊 STRATEGY COMPARISON")
print("=" * 80 + "\n")

print("⏱️  Execution Time:")
print(f"  Strategy 1: {elapsed_s1:6.2f}s")
print(f"  Strategy 2: {elapsed_s2:6.2f}s")
print(f"  Strategy 3: {elapsed_s3:6.2f}s")
print(f"  Total:      {elapsed_s1 + elapsed_s2 + elapsed_s3:6.2f}s")

print("\n📈 Fields Added:")
if 'result_df_s1' in locals():
    s1_fields = [col for col in result_df_s1.columns if col not in df.columns]
    print(f"  Strategy 1: {len(s1_fields)} fields ({', '.join(s1_fields)})")

if 'result_df_s2' in locals():
    s2_fields = [col for col in result_df_s2.columns if col not in df.columns]
    print(f"  Strategy 2: {len(s2_fields)} fields ({', '.join(s2_fields)})")

if 'result_df_s3' in locals():
    s3_fields = [col for col in result_df_s3.columns if col not in df.columns]
    print(f"  Strategy 3: {len(s3_fields)} fields ({', '.join(s3_fields)})")

## Step 6: Detailed Artifact Review

**How to use:**
- Run for comprehensive artifact inspection
- Focus on validation points per strategy

**What you'll see (per strategy):**
1. **Metrics JSON**: Field counts, correlations, statistics, performance
2. **Visualizations**: Charts displayed inline (latest batch only)

**Note:** Only newest files shown to avoid confusion from multiple runs

In [ ]:
# Review all strategies
strategy_dirs = [
    ('strategy1_lookup', 'Strategy 1: Lookup-Based'),
    ('strategy2_conditional', 'Strategy 2: Conditional'),
    ('strategy3_expression', 'Strategy 3: Expression')
]

for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    if not strategy_dir.exists():
        continue
        
    print("\n" + "=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    # 1. Metrics (NEWEST - with filtering)
    metrics_dir = strategy_dir / 'metrics'
    if metrics_dir.exists():
        metrics_files = list(metrics_dir.glob('*.json'))
        
        if metrics_files:
            # Exclude data_types files
            filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

            if filtered:
                target_files = filtered
                print(f"✓ Found {len(filtered)} metrics file(s)")
            else:
                target_files = metrics_files
                print(f"⚠ Fallback to ALL {len(metrics_files)} file(s)")

            # Pick latest
            target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
            latest_metrics_file = target_files[0]
            
            print(f"📄 {latest_metrics_file.name}")
            
            try:
                with open(latest_metrics_file, 'r') as f:
                    data = json.load(f)
                    metrics = data.get('metrics', {})
                
                # Display key metrics
                print(f"\n   Fields Added: {metrics.get('fields_added_count', 0)}")
                print(f"   Fields Modified: {metrics.get('fields_modified_count', 0)}")
                
                if 'execution_time_seconds' in metrics:
                    print(f"   Execution Time: {metrics['execution_time_seconds']:.4f}s")
                    
                if 'records_processed' in metrics:
                    print(f"   Records Processed: {metrics['records_processed']:,}")
                    
                # Show correlations if available
                if 'correlations' in metrics and metrics['correlations']:
                    print(f"\n   Correlations:")
                    for field, corr in list(metrics['correlations'].items())[:3]:
                        if not pd.isna(corr):
                            print(f"      {field}: {corr:.3f}")
                    
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
    
    # 2. Visualizations (NEWEST BATCH)
    viz_dir = strategy_dir / 'visualizations'
    if viz_dir.exists():
        viz_files = sorted(
            list(viz_dir.glob('*.png')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            latest_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < 10
            ]
            
            print(f"\n📊 VISUALIZATIONS: {len(latest_batch)} files")
            for viz in latest_batch[:2]:  # Show first 2 only
                print(f"   📈 {viz.stem.replace('_', ' ').title()}")
                try:
                    display(Image(filename=str(viz)))
                except:
                    print(f"      (Display error)")

## Step 7: Calculate Enrichment Metrics

**How to use:**
- Calculate data enrichment improvements
- Run AFTER all 3 strategies complete

**What you'll see:**
- Original data: column count, completeness
- After enrichment: added columns, enhanced metrics
- Enrichment ratios (e.g., 60% more fields)

**Enrichment targets:**
- 30%+ field increase: Good enrichment
- 50%+ field increase: Strong enrichment
- 100%+ field increase: Comprehensive enrichment

**Note:** Strategy 1 (lookup) provides fastest enrichment from external data

In [ ]:
print("\n" + "=" * 80)
print("📊 DATA ENRICHMENT METRICS")
print("=" * 80 + "\n")

def calculate_enrichment_score(original_df, enriched_df):
    """Calculate enrichment metrics"""
    orig_cols = len(original_df.columns)
    enriched_cols = len(enriched_df.columns)
    added_cols = enriched_cols - orig_cols
    
    return {
        'original_columns': orig_cols,
        'enriched_columns': enriched_cols,
        'added_columns': added_cols,
        'enrichment_ratio': (added_cols / orig_cols) if orig_cols > 0 else 0
    }

# Original data
print(f"📊 ORIGINAL DATA:")
print(f"   Columns: {len(df.columns)}")
print(f"   Records: {len(df):,}")

# Strategy 1 enrichment
if 'result_df_s1' in locals():
    s1_metrics = calculate_enrichment_score(df, result_df_s1)
    print(f"\n✨ STRATEGY 1 (Lookup-Based):")
    print(f"   Added: {s1_metrics['added_columns']} columns")
    print(f"   Enrichment: {s1_metrics['enrichment_ratio']:.1%}")

# Strategy 2 enrichment
if 'result_df_s2' in locals():
    s2_metrics = calculate_enrichment_score(df, result_df_s2)
    print(f"\n✨ STRATEGY 2 (Conditional):")
    print(f"   Added: {s2_metrics['added_columns']} columns")
    print(f"   Enrichment: {s2_metrics['enrichment_ratio']:.1%}")

# Strategy 3 enrichment
if 'result_df_s3' in locals():
    s3_metrics = calculate_enrichment_score(df, result_df_s3)
    print(f"\n✨ STRATEGY 3 (Expression):")
    print(f"   Added: {s3_metrics['added_columns']} columns")
    print(f"   Enrichment: {s3_metrics['enrichment_ratio']:.1%}")

# Combined enrichment
if all(k in locals() for k in ['result_df_s1', 'result_df_s2', 'result_df_s3']):
    total_added = s1_metrics['added_columns'] + s2_metrics['added_columns'] + s3_metrics['added_columns']
    total_ratio = (total_added / len(df.columns)) if len(df.columns) > 0 else 0
    print(f"\n🎯 TOTAL ENRICHMENT:")
    print(f"   Total Added: {total_added} columns")
    print(f"   Overall Enrichment: {total_ratio:.1%}")

## Step 8: Export Final Data

**How to use:**
- Export enriched datasets from all strategies
- Run AFTER all 3 strategies complete

**What you'll see (per strategy):**
- Available columns list (original + added)
- Export confirmation (file path, column/row count)
- Preview of first 5 rows
- Field statistics (unique values per new field)

**Output structure:**
```
advanced_output/
├── strategy1_lookup/lookup_enriched_data.csv
├── strategy2_conditional/conditional_enriched_data.csv
└── strategy3_expression/expression_enriched_data.csv
```

**Note:** Files include both original and enriched fields for comparison

In [ ]:
import os
from pathlib import Path

print("=" * 80)
print("📦 EXPORTING FINAL DATA FROM ALL STRATEGIES")
print("=" * 80)

# Base export directory
export_base_dir = task_dir
task_dir.mkdir(parents=True, exist_ok=True)

print(f"\n📂 Export directory: {task_dir}\n")

# ============================================================================
# STRATEGY 1: Lookup-Based Enrichment
# ============================================================================
if 'result_df_s1' in locals() and result_df_s1 is not None:
    print("=" * 80)
    print("📊 STRATEGY 1: LOOKUP-BASED ENRICHMENT")
    print("=" * 80)
    
    s1_dir = export_base_dir / 'strategy1_lookup'
    s1_dir.mkdir(parents=True, exist_ok=True)
    
    # Save
    output_path_s1 = s1_dir / 'lookup_enriched_data.csv'
    try:
        result_df_s1.to_csv(output_path_s1, index=False)
        print(f"\n✅ Saved: {output_path_s1}")
        print(f"   Columns: {len(result_df_s1.columns)}")
        print(f"   Rows: {len(result_df_s1):,}")
    except PermissionError:
        print(f"\n⚠️  Cannot save (file may be open)")
    
    # Preview
    print(f"\n📊 Preview:")
    new_cols = [col for col in result_df_s1.columns if col not in df.columns]
    display_cols = list(df.columns[:3]) + new_cols
    print(result_df_s1[display_cols].head())
    
    # Statistics
    print(f"\n📈 New Field Statistics:")
    for col in new_cols:
        print(f"   {col}: {result_df_s1[col].nunique()} unique values")
else:
    print("\n⚠️  Strategy 1 data not available")

# ============================================================================
# STRATEGY 2: Conditional Logic
# ============================================================================
if 'result_df_s2' in locals() and result_df_s2 is not None:
    print("\n" + "=" * 80)
    print("📊 STRATEGY 2: CONDITIONAL LOGIC")
    print("=" * 80)
    
    s2_dir = export_base_dir / 'strategy2_conditional'
    s2_dir.mkdir(parents=True, exist_ok=True)
    
    output_path_s2 = s2_dir / 'conditional_enriched_data.csv'
    try:
        result_df_s2.to_csv(output_path_s2, index=False)
        print(f"\n✅ Saved: {output_path_s2}")
        print(f"   Columns: {len(result_df_s2.columns)}")
        print(f"   Rows: {len(result_df_s2):,}")
    except PermissionError:
        print(f"\n⚠️  Cannot save (file may be open)")
    
    print(f"\n📊 Preview:")
    new_cols = [col for col in result_df_s2.columns if col not in df.columns]
    display_cols = list(df.columns[:3]) + new_cols
    print(result_df_s2[display_cols].head())
    
    print(f"\n📈 New Field Statistics:")
    for col in new_cols:
        print(f"   {col}: {result_df_s2[col].nunique()} unique values")
else:
    print("\n⚠️  Strategy 2 data not available")

# ============================================================================
# STRATEGY 3: Expression-Based Calculations
# ============================================================================
if 'result_df_s3' in locals() and result_df_s3 is not None:
    print("\n" + "=" * 80)
    print("📊 STRATEGY 3: EXPRESSION-BASED CALCULATIONS")
    print("=" * 80)
    
    s3_dir = export_base_dir / 'strategy3_expression'
    s3_dir.mkdir(parents=True, exist_ok=True)
    
    output_path_s3 = s3_dir / 'expression_enriched_data.csv'
    try:
        result_df_s3.to_csv(output_path_s3, index=False)
        print(f"\n✅ Saved: {output_path_s3}")
        print(f"   Columns: {len(result_df_s3.columns)}")
        print(f"   Rows: {len(result_df_s3):,}")
    except PermissionError:
        print(f"\n⚠️  Cannot save (file may be open)")
    
    print(f"\n📊 Preview:")
    new_cols = [col for col in result_df_s3.columns if col not in df.columns]
    display_cols = list(df.columns[:3]) + new_cols
    print(result_df_s3[display_cols].head())
    
    print(f"\n📈 New Field Statistics:")
    for col in new_cols:
        if pd.api.types.is_numeric_dtype(result_df_s3[col]):
            print(f"   {col}: mean=${result_df_s3[col].mean():.2f}, range=${result_df_s3[col].min():.2f}-${result_df_s3[col].max():.2f}")
        else:
            print(f"   {col}: {result_df_s3[col].nunique()} unique values")
else:
    print("\n⚠️  Strategy 3 data not available")

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "=" * 80)
print("✅ EXPORT COMPLETE")
print("=" * 80)
print(f"\n📂 All files saved to: {export_base_dir}")

if 'elapsed_s1' in locals():
    total_time = elapsed_s1 + elapsed_s2 + elapsed_s3
    print(f"⏱️  Total time: {total_time:.2f}s")

## 🎯 Summary

**Accomplished:**
- ✅ 3 field operation strategies implemented and compared
- ✅ Data enrichment metrics calculated
- ✅ Performance benchmarking completed
- ✅ Production-ready artifacts generated

**Key takeaways:**
- Lookup-based enrichment fastest for external reference data
- Conditional logic enables complex business rules
- Expression-based calculations derive meaningful KPIs

**Strategy recommendations:**
- **Use Strategy 1** when: Enriching from external dictionaries or master data
- **Use Strategy 2** when: Implementing business logic or dynamic categorization
- **Use Strategy 3** when: Creating calculated metrics or derived KPIs

**Next steps:**
- Test with your own datasets and lookup tables
- Combine strategies for comprehensive enrichment
- Deploy to production data pipelines

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)